In [1]:
"""
Network Topology Feature Extraction (Simplified)
=================================================
从Reddit互动网络提取拓扑特征

输入: reddit_wsb_for_network.csv
输出: network_features_5min.parquet (与GME 5分钟bars对齐)

核心特征:
1. Degree Centrality (连接数)
2. Betweenness Centrality (桥梁作用)
3. K-Core Number (核心-边缘结构)
4. Clustering Coefficient (聚类系数)
5. PageRank (影响力)

Requirements:
    pip install networkx
"""

import pandas as pd
import numpy as np
import networkx as nx
from datetime import datetime, timedelta
from collections import defaultdict

# ============================================================
# 参数设置
# ============================================================

TIME_WINDOW = 5  # 分钟
NETWORK_FILE = 'reddit_wsb_for_network.csv'

# ============================================================
# 核心函数
# ============================================================

def build_interaction_network(data_window):
    """
    构建时间窗口内的用户互动网络
    
    边的类型:
    1. Reply: A replies to B
    2. Mention: A mentions B in post/comment
    """
    
    G = nx.DiGraph()  # 有向图
    
    for _, row in data_window.iterrows():
        user = row['user_id']
        
        # 添加节点
        if user not in G:
            G.add_node(user)
        
        # Reply edge
        if pd.notna(row['reply_to_user']) and row['reply_to_user'] != user:
            reply_to = row['reply_to_user']
            if reply_to not in G:
                G.add_node(reply_to)
            
            # 添加或更新边
            if G.has_edge(user, reply_to):
                G[user][reply_to]['weight'] += 1
            else:
                G.add_edge(user, reply_to, weight=1, type='reply')
        
        # Mention edges
        if isinstance(row['mentions'], list) and len(row['mentions']) > 0:
            for mentioned in row['mentions']:
                if mentioned != user:
                    if mentioned not in G:
                        G.add_node(mentioned)
                    
                    if G.has_edge(user, mentioned):
                        G[user][mentioned]['weight'] += 1
                    else:
                        G.add_edge(user, mentioned, weight=1, type='mention')
    
    return G


def extract_network_features(G):
    """
    从网络中提取拓扑特征
    """
    
    if G.number_of_nodes() == 0:
        return {
            'node_count': 0,
            'edge_count': 0,
            'density': 0,
            'avg_degree': 0,
            'max_degree': 0,
            'avg_betweenness': 0,
            'max_betweenness': 0,
            'avg_clustering': 0,
            'max_k_core': 0,
            'avg_pagerank': 0,
            'max_pagerank': 0,
            'num_components': 0,
            'largest_component_size': 0
        }
    
    # 基础指标
    features = {
        'node_count': G.number_of_nodes(),
        'edge_count': G.number_of_edges()
    }
    
    # 密度
    features['density'] = nx.density(G)
    
    # Degree centrality
    degrees = dict(G.degree())
    if degrees:
        features['avg_degree'] = np.mean(list(degrees.values()))
        features['max_degree'] = max(degrees.values())
    else:
        features['avg_degree'] = 0
        features['max_degree'] = 0
    
    # Betweenness centrality (计算密集，只在小网络上计算)
    if G.number_of_nodes() < 1000:
        try:
            betweenness = nx.betweenness_centrality(G)
            features['avg_betweenness'] = np.mean(list(betweenness.values()))
            features['max_betweenness'] = max(betweenness.values())
        except:
            features['avg_betweenness'] = 0
            features['max_betweenness'] = 0
    else:
        # 大网络跳过（太慢）
        features['avg_betweenness'] = 0
        features['max_betweenness'] = 0
    
    # Clustering coefficient (转为无向图)
    try:
        G_undirected = G.to_undirected()
        clustering = nx.clustering(G_undirected)
        features['avg_clustering'] = np.mean(list(clustering.values()))
    except:
        features['avg_clustering'] = 0
    
    # K-core
    try:
        G_undirected = G.to_undirected()
        k_core = nx.core_number(G_undirected)
        features['max_k_core'] = max(k_core.values()) if k_core else 0
    except:
        features['max_k_core'] = 0
    
    # PageRank
    try:
        pagerank = nx.pagerank(G, max_iter=50)
        features['avg_pagerank'] = np.mean(list(pagerank.values()))
        features['max_pagerank'] = max(pagerank.values())
    except:
        features['avg_pagerank'] = 0
        features['max_pagerank'] = 0
    
    # Connected components (弱连通分量)
    try:
        components = list(nx.weakly_connected_components(G))
        features['num_components'] = len(components)
        features['largest_component_size'] = max(len(c) for c in components) if components else 0
    except:
        features['num_components'] = 0
        features['largest_component_size'] = 0
    
    return features


# ============================================================
# 主函数
# ============================================================

def main():
    start_time = datetime.now()
    
    print("\n" + "="*70)
    print("NETWORK TOPOLOGY FEATURE EXTRACTION")
    print("="*70)
    
    # ========== Step 1: 加载数据 ==========
    print("\nStep 1: Loading Reddit network data...")
    
    try:
        reddit_df = pd.read_csv(NETWORK_FILE)
        reddit_df['timestamp'] = pd.to_datetime(reddit_df['timestamp'])
        
        # 处理mentions列（可能是字符串）
        if 'mentions' in reddit_df.columns:
            def parse_mentions(x):
                if pd.isna(x) or x == '' or x == '[]':
                    return []
                try:
                    # 尝试作为Python list解析
                    import ast
                    return ast.literal_eval(x) if isinstance(x, str) else x
                except:
                    return []
            
            reddit_df['mentions'] = reddit_df['mentions'].apply(parse_mentions)
        else:
            reddit_df['mentions'] = [[] for _ in range(len(reddit_df))]
        
        print(f"  ✓ Loaded {len(reddit_df):,} items")
        print(f"  Date range: {reddit_df['timestamp'].min()} to {reddit_df['timestamp'].max()}")
        
    except FileNotFoundError:
        print(f"  ✗ {NETWORK_FILE} not found!")
        return
    
    # ========== Step 2: 创建时间窗口 ==========
    print("\nStep 2: Creating time windows...")
    
    reddit_df['time_window'] = reddit_df['timestamp'].dt.floor(f'{TIME_WINDOW}min')
    time_windows = sorted(reddit_df['time_window'].unique())
    
    print(f"  ✓ Created {len(time_windows):,} time windows")
    
    # ========== Step 3: 逐窗口提取网络特征 ==========
    print("\nStep 3: Extracting network features per window...")
    
    features_list = []
    
    for i, window_time in enumerate(time_windows):
        if i % 100 == 0:
            print(f"  Processing window {i+1:,}/{len(time_windows):,}...", end='\r')
        
        # 获取该窗口的数据
        window_data = reddit_df[reddit_df['time_window'] == window_time]
        
        # 构建网络
        G = build_interaction_network(window_data)
        
        # 提取特征
        window_features = extract_network_features(G)
        window_features['timestamp'] = window_time
        
        features_list.append(window_features)
    
    print(f"\n  ✓ Extracted features for {len(features_list):,} windows")
    
    # 转换为DataFrame
    network_df = pd.DataFrame(features_list)
    
    # 基础统计
    print(f"\n  Network statistics:")
    print(f"    Mean nodes: {network_df['node_count'].mean():.1f}")
    print(f"    Mean edges: {network_df['edge_count'].mean():.1f}")
    print(f"    Mean density: {network_df['density'].mean():.4f}")
    print(f"    Mean max degree: {network_df['max_degree'].mean():.1f}")
    
    # ========== Step 4: 保存 ==========
    print("\nStep 4: Saving...")
    
    network_df.to_parquet('network_features_5min.parquet', index=False)
    
    print(f"  ✓ Saved: network_features_5min.parquet")
    print(f"  Shape: {network_df.shape}")
    
    # ========== 总结 ==========
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    
    print("\n" + "="*70)
    print("COMPLETE")
    print("="*70)
    print(f"\nTime windows: {len(network_df):,}")
    print(f"Feature columns: {len(network_df.columns)}")
    
    print(f"\nFeatures extracted:")
    print(f"  ✓ node_count, edge_count, density")
    print(f"  ✓ avg_degree, max_degree")
    print(f"  ✓ avg_betweenness, max_betweenness")
    print(f"  ✓ avg_clustering")
    print(f"  ✓ max_k_core")
    print(f"  ✓ avg_pagerank, max_pagerank")
    print(f"  ✓ num_components, largest_component_size")
    
    print(f"\nRuntime: {duration:.1f} seconds")
    
    print("\n✓ Feature extraction complete!")
    print("  (Alignment will be done in merge_all_features.py)")


if __name__ == "__main__":
    main()
    


NETWORK TOPOLOGY FEATURE EXTRACTION

Step 1: Loading Reddit network data...
  ✓ Loaded 1,606,093 items
  Date range: 2019-07-01 04:35:02 to 2021-06-29 23:58:28

Step 2: Creating time windows...
  ✓ Created 77,267 time windows

Step 3: Extracting network features per window...
  Processing window 77,201/77,267...
  ✓ Extracted features for 77,267 windows

  Network statistics:
    Mean nodes: 21.7
    Mean edges: 7.1
    Mean density: 0.0540
    Mean max degree: 3.5

Step 4: Saving...
  ✓ Saved: network_features_5min.parquet
  Shape: (77267, 14)

COMPLETE

Time windows: 77,267
Feature columns: 14

Features extracted:
  ✓ node_count, edge_count, density
  ✓ avg_degree, max_degree
  ✓ avg_betweenness, max_betweenness
  ✓ avg_clustering
  ✓ max_k_core
  ✓ avg_pagerank, max_pagerank
  ✓ num_components, largest_component_size

Runtime: 402.8 seconds

✓ Feature extraction complete!
  (Alignment will be done in merge_all_features.py)
